# LogVerse AI Platform — Agentic Log Intelligence & Medallion Pipeline
**An Enterprise-Grade Agentic AI Platform for Log Intelligence, Anomaly Detection, Medallion Architecture, and Root Cause Analysis.**

This notebook demonstrates the end-to-end pipeline:
1. **Medallion Lakehouse Architecture** (Bronze -> Silver -> Gold)
2. **PyTorch Transformer Backbone Anomaly Detection** (`transformer_backbone.pth`)
3. **Multi-Agent AI Orchestration** (Planner, Catalog, ML, RCA, SLM, Remediation Agents)
4. **Interactive Causal Root Cause Analysis (RCA) Graph Generation**
5. **Local SLM Human-Understandable Diagnostics & Operational Remediation**

In [ ]:
# 1. Imports and System Initialization
import os
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from logverse_pipeline import MedallionPipeline, generate_sample_hdfs_log
from logverse_ml import MLAnomalyDetector
from logverse_rca_graph import RCAGraphBuilder
from logverse_agents import MultiAgentOrchestrator

print("✓ LogVerse AI Modules Loaded Successfully!")

## Step 1: Execute Medallion Lakehouse Data Pipeline
Raw log files pass through **Bronze** (Immutable Raw Store), **Silver** (Parsed, Regex Template Matching & Sessionization), and **Gold** (AI Feature Aggregation & Dataset Indexing) layers.

In [ ]:
sample_log_data = generate_sample_hdfs_log()
pipeline = MedallionPipeline(source_type="HDFS")
bronze, silver_df, gold_df, ai_catalog = pipeline.process_raw_logs(sample_log_data, source_name="sample_hdfs.log")

print("--- AI CATALOG METADATA ---")
for k, v in ai_catalog.items():
    print(f"{k}: {v}")

print("\n--- GOLD LAYER SESSIONS ---")
display(gold_df)

## Step 2: PyTorch Transformer Backbone Anomaly Detection
Invoking the trained `TransformerModel` backbone to extract sequence feature embeddings and evaluate anomaly probabilities.

In [ ]:
ml_detector = MLAnomalyDetector()
print("ML Model Weights Loaded:", ml_detector.loaded)

for idx, row in gold_df.iterrows():
    blk = row["BlockId"]
    res = ml_detector.predict_session(row["EventList"])
    print(f"\nBlock: {blk}")
    print(f"  Status: {res['status_label']}")
    print(f"  Anomaly Prob: {res['anomaly_probability']:.1%}")
    print(f"  Suspicious Events: {res['suspicious_events']}")

## Step 3: Run Multi-Agent Orchestrator
Watch 6 AI Agents (Planner, Catalog, ML, RCA Graph, SLM Diagnostic, Remediation) collaborate to analyze logs and produce explainable insights.

In [ ]:
orchestrator = MultiAgentOrchestrator()
agent_results = orchestrator.run_agentic_pipeline(sample_log_data)

print("=== AGENT COMMUNICATION LOG ===")
for msg in agent_results["communication_log"]:
    print(f"[{msg['timestamp']}] {msg['sender']} ➔ {msg['recipient']}: ({msg['action']}) {msg['content']}")

## Step 4: Causal Root Cause Analysis (RCA) Graph Output
Visualizing fault propagation from System components to anomalous events and pinpointing the primary Root Cause node.

In [ ]:
graph_builder = RCAGraphBuilder()
target_blk = agent_results["target_block"]
target_recs = agent_results["silver_df"][agent_results["silver_df"]["BlockId"] == target_blk].to_dict(orient="records")
graph, root_causes = graph_builder.build_graph_from_session(target_blk, target_recs, agent_results["ml_results"].get(target_blk, {}))

plt.figure(figsize=(12, 6))
pos = nx.spring_layout(graph, seed=42)
colors = [data.get("color", "#3b82f6") for _, data in graph.nodes(data=True)]
nx.draw(graph, pos, with_labels=True, node_color=colors, font_color="white", font_weight="bold", node_size=2000, font_size=8)
plt.title(f"Causal Root Cause Analysis Graph for {target_blk}", fontsize=14)
plt.show()

print("Identified Primary Root Causes:", root_causes)

## Step 5: Local SLM Human-Understandable Diagnosis & Remediation
Local SLM generates clear explanations for operators along with step-by-step SOP remediation runbooks.

In [ ]:
slm = agent_results["slm_explanation"]
rem = agent_results["remediation"]

print("=== SLM HUMAN DIAGNOSIS ===")
print("Summary:", slm["summary"])
print("\nMechanism:", slm["mechanism"])
print("\nImpact:", slm["impact"])
print("\nConfidence:", slm["confidence_note"])

print("\n=== OPERATIONAL REMEDIATION SOP ===")
print("Urgency:", rem["urgency"])
for step in rem["sop_steps"]:
    print(step)